# LLM clients

This notebook demonstrates the usage of Kaval.AI LLM clients.

In [1]:
# Load dotenv
import dotenv

dotenv.load_dotenv("../.env")

True

## Basic prompting and text generation

In [2]:
from kavalai.llm_clients.v2.gemini_client import GeminiClient
from kavalai.llm_clients.v2.openai_client import OpenAIClient

gemini_client = GeminiClient(model="gemini-3.1-flash-lite")
openai_client = OpenAIClient(model="gpt-5.4-mini")

/home/timo/projects/kaval.ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
gemini_result = await gemini_client.prompt("Start a conversation with OpenAI, say who you are. Max 50 words.")
print(f"Gemini: {gemini_result}\n")

openai_result = await openai_client.prompt(f"Gemini: f{gemini_result}")
print(f"OpenAI: {openai_result}\n")

2026-06-04 23:24:56.333 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-3.1-flash-lite): 66 tokens, 0.68s


Gemini: Hello, OpenAI. I am an AI assistant designed to provide helpful, accurate, and creative information. I am reaching out to initiate a dialogue, explore potential synergies, and learn from your advanced language models. I look forward to our interaction.



2026-06-04 23:24:59.586 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-5.4-mini): 140 tokens, 3.25s


OpenAI: Hello! Nice to meet you. I’m happy to chat and collaborate.

I can help with things like:
- answering questions
- brainstorming ideas
- writing and editing
- coding and debugging
- summarizing and explaining concepts

If you’d like, we can start with a topic you’re interested in, or I can ask you a few questions to tailor the conversation.



## Structured outputs

In [4]:
from pydantic import BaseModel

html = """
<table>
  <tr>
    <th>Company</th>
    <th>Contact</th>
    <th>Location</th>
  </tr>
  <tr>
    <td>Kaval.AI</td>
    <td>Timo Petmanson</td>
    <td>Estonia</td>
  </tr>
  <tr>
    <td>Intergalactic Government</td>
    <td>😼</td>
    <td>Jupiter</td>
  </tr>
</table>
"""


class Company(BaseModel):
    name: str
    contact_info: str
    country: str


class ParseResult(BaseModel):
    companies: list[Company]


companies = await gemini_client.prompt(f"Parse company data from HTML\n{html}",
                                       response_model=ParseResult)

print(companies)


2026-06-04 23:25:00.454 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-3.1-flash-lite): 119 tokens, 0.62s


companies=[Company(name='Kaval.AI', contact_info='Timo Petmanson', country='Estonia'), Company(name='Intergalactic Government', contact_info='😼', country='Jupiter')]


## Streaming

In [5]:
streamer = await openai_client.stream_prompt("Count to 3. Separate values by commas.")
async for chunk in streamer:
    print(chunk)


type='partial' name='response' value='1'
type='partial' name='response' value='1,'
type='partial' name='response' value='1, '
type='partial' name='response' value='1, 2'
type='partial' name='response' value='1, 2,'
type='partial' name='response' value='1, 2, '
type='partial' name='response' value='1, 2, 3'


2026-06-04 23:25:02.022 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (openai/gpt-5.4-mini): 27 tokens, 1.42s


type='complete' name='response' value='1, 2, 3'


Streaming structured data always returns valid JSON for partial responses.

In [6]:
class UserProfile(BaseModel):
    name: str
    birthdate: str
    skills: list[str]
    hobbies: list[str]


streamer = await gemini_client.stream_prompt("Generate a mock user profile",
                                             response_model=UserProfile)
async for chunk in streamer:
    print(chunk)

2026-06-04 23:25:02.868 | INFO     | kavalai.llm_clients.base_client:receive_model_stats:82 - Model stats (gemini/gemini-3.1-flash-lite): 87 tokens, 0.64s


type='partial' name='response' value='{"name": "Elena"}'
type='partial' name='response' value='{"name": "Elena Rodriguez", "birthdate": "1992-05-14"}'
type='partial' name='response' value='{"name": "Elena Rodriguez", "birthdate": "1992-05-14", "skills": ["Python", "Data Analysis", "Project Management"]}'
type='partial' name='response' value='{"name": "Elena Rodriguez", "birthdate": "1992-05-14", "skills": ["Python", "Data Analysis", "Project Management"], "hobbies": ["Hiking", "Photography", "Cooking"]}'
type='complete' name='response' value='{"name": "Elena Rodriguez", "birthdate": "1992-05-14", "skills": ["Python", "Data Analysis", "Project Management"], "hobbies": ["Hiking", "Photography", "Cooking"]}'


## Model call stats

By default, KavalAI logs the call stats using `loguru` and `ModelStatsLogger` class.
But you can provide your custom stats receiver and implement custom logging behavior.

In [13]:
from kavalai.llm_clients.base_client import ModelStatsReceiver, ModelCallStat

class MyStatReceiver(ModelStatsReceiver):
    def receive_model_stats(self, stats: ModelCallStat):
        print(f"Received model stats: {stats.model_dump_json(indent=2)}")

gemini_client = GeminiClient(model="gemini-3.1-flash-lite", model_stats_receiver=MyStatReceiver())

result = await gemini_client.prompt("What is 2+2?")
print(f"Gemini: {result}\n")

Received model stats: {
  "call_type": "llm",
  "model": "gemini/gemini-3.1-flash-lite",
  "request_data": "{\"model\": \"gemini-3.1-flash-lite\", \"contents\": [\"parts=[Part(\\n  text='What is 2+2?'\\n)] role='user'\"], \"config\": {\"temperature\": 1.0, \"top_p\": 0.2}}",
  "response_data": "2 + 2 = 4",
  "response_code": null,
  "prompt_tokens": 8,
  "completion_tokens": 7,
  "total_tokens": 15,
  "batch_size": null,
  "duration_seconds": 0.5076468599982036
}
Gemini: 2 + 2 = 4

